In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import project_helpers_metrics as phm
import project_helpers_ga as phg

In [2]:
#testing daily_returns function
test = pd.DataFrame({"stock_a": [100, 110, 99, 99]})
print(phm.daily_returns(test))

   stock_a
1      0.1
2     -0.1
3      0.0


In [3]:
#tests reading real data
prices = pd.read_csv("data/data.csv", index_col=0, parse_dates=True)
returns = phm.daily_returns(prices)
print(returns.shape)
print(returns.head(3))

(5282, 7)
                 EEM       EFA       GLD       IWM       SPY       TLT  \
Date                                                                     
2005-10-04 -0.009365 -0.006563  0.000646 -0.015122 -0.011256  0.003292   
2005-10-05 -0.030371 -0.008693 -0.001938 -0.025540 -0.013116  0.004376   
2005-10-06 -0.034979 -0.004735  0.018335 -0.008892 -0.003595 -0.003812   

                 VNQ  
Date                  
2005-10-04 -0.016924  
2005-10-05 -0.015527  
2005-10-06 -0.001372  


In [4]:
#prints the test2 dataframe for portfolio_returns
test2 = pd.DataFrame({"A": [0.10, 0.00], "B": [-0.05, 0.20]})
print(test2)

     A     B
0  0.1 -0.05
1  0.0  0.20


In [5]:
#tests weights with fake weights
print(phm.portfolio_returns([0.6, 0.4], test2))

0    0.04
1    0.08
dtype: float64


In [6]:
#tests annual returns
test3 = pd.Series([0.001, 0.002, 0.003])
print(phm.annual_return(test3))

0.504


In [7]:
#tests annual volatility
stable = pd.Series([0.001, 0.001, 0.001, 0.001])
volatile = pd.Series([0.01, -0.01, 0.01, -0.01])
print(phm.annual_volatility(stable))
print(phm.annual_volatility(volatile))

0.0
0.18330302779823363


In [8]:
#tests the sharpe ratio
print(phm.sharpe(volatile))

0.0


In [9]:
#tests the normalize function for normal case
print(phg.normalize(np.array([2, -1, 3])))

[0.4 0.  0.6]


In [10]:
#tests the normalize function for an equal case
print(phg.normalize(np.array([1, 1, 1, 1])))

[0.25 0.25 0.25 0.25]


In [11]:
#tests the normalize functions for a negative case
print(phg.normalize(np.array([-1, -2, -3])))

[0.33333333 0.33333333 0.33333333]


In [12]:
#tests random portfolio function
np.random.seed(6449)
p = phg.random_portfolio(7) # 7 markets chosen
print(p)
print(len(p), p.sum(), (p >= 0).all())

[0.00864115 0.05201257 0.19318486 0.05106355 0.09383036 0.24139008
 0.35987744]
7 0.9999999999999999 True


In [13]:
#tests the make_population function. 5 portfolios of 7 to make sure each one adds up to 1
np.random.seed(6449) #makes a random generated set with bu id
pop = phg.make_population(5 # number of portolios 
                        , 7) # how many funds in each portfolio (5x7 grid)
print(len(pop))
for p in pop:
    print(p.round(3), p.sum().round(3))

5
[0.009 0.052 0.193 0.051 0.094 0.241 0.36 ] 1.0
[0.141 0.312 0.054 0.087 0.203 0.007 0.196] 1.0
[0.275 0.231 0.012 0.114 0.088 0.159 0.123] 1.0
[0.209 0.069 0.173 0.039 0.18  0.212 0.117] 1.0
[0.203 0.199 0.149 0.135 0.099 0.129 0.086] 1.0


In [14]:
#test to see if the fitness function works properly using the dates used for the training period
train = returns.loc['2005-10-03':'2019-12-31']
print(train.shape)

equal = np.ones(7)/7
spy_only = np.array([0, 0, 0, 0, 1, 0, 0])

print('equal:', phg.fitness(equal, train))
print('SPY:', phg.fitness(spy_only, train))

(3585, 7)
equal: 0.5870602319528152
SPY: 0.5654212926520932


In [15]:
#test for select function which it should keep 2 of the top scores as parents for training
test_pop = ['A', 'B', 'C', 'D']
test_scores = np.array([0.3, 0.9, 0.1, 0.6])
print(phg.select(test_pop, test_scores, 2))

['B', 'D']


In [16]:
#tests crossover function for identical parents
same = np.array([0.5, 0.3, 0.2])
print(phg.crossover(same, same))

[0.5 0.3 0.2]


In [17]:
#tests different parents for crossover function with bu id
np.random.seed(6449)
a = np.array([0.4, 0.3, 0.2, 0.1])
b = np.array([0.1, 0.2, 0.3, 0.4])

child = phg.crossover(a,b)
print(child, len(child), child.sum(), (child >= 0).all())

[0.28571429 0.21428571 0.21428571 0.28571429] 4 1.0 True


In [18]:
#tests mutate function with rate of 0 (never mutates)
never = np.array([0.5, 0.3, 0.2])
print(phg.mutate(never, 0))

[0.5 0.3 0.2]


In [19]:
#tests mutate function with rate of 1 where it always mutates
np.random.seed(6449)
port = np.array([0.5, 0.3, 0.2])
m = phg.mutate(port, 1)

print('mutated:', m, len(m), m.sum(), (m >= 0).all())
print('original:', port)

mutated: [0.29438708 0.17663225 0.52898067] 3 0.9999999999999998 True
original: [0.5 0.3 0.2]


In [20]:
#testing for breeding 
best, hist = phg.run_ga(train, pop_size=20, n_keep = 5, generations = 10, rate= 0.1, seed= 6449)
for h in hist:
    print('sharpe ratio:', round(h,4))

sharpe ratio: 0.7799
sharpe ratio: 0.8333
sharpe ratio: 0.9208
sharpe ratio: 0.9347
sharpe ratio: 0.9822
sharpe ratio: 0.9822
sharpe ratio: 1.0022
sharpe ratio: 1.0022
sharpe ratio: 1.0025
sharpe ratio: 1.0026


In [21]:
#tests for final scoring 
best, hist = phg.run_ga(train, pop_size = 20, n_keep = 5, generations = 10, rate = 0.1, seed = 6449)

print(pd.Series(best, index = train.columns).map('{:.1%}'.format)) #googled the percentage
print(best.sum(), (best >= 0).all())
print('final fitness:', phg.fitness(best, train), 'best in last gen:', hist[-1])#prints the last in place aka the best score

EEM     0.8%
EFA     2.5%
GLD    23.2%
IWM     2.0%
SPY    32.5%
TLT    36.4%
VNQ     2.7%
dtype: str
0.9999999999999999 True
final fitness: 1.002553170137426 best in last gen: 1.002553170137426


In [22]:
#tests the max_drawdown function with worst drop
print(phm.max_drawdown(pd.Series([0.2, -0.25, 0.10, -0.5])))

-0.5874999999999999


In [23]:
#tests max_drawdown if it never falls below the peak
print(phm.max_drawdown(pd.Series([0.01, 0.02, 0.03])))

0.0


In [24]:
#tests the ga with 2 funds
two_fund = train[['SPY', 'TLT']]
print(two_fund.shape)

(3585, 2)


In [25]:
# brute-force test to score every split from 0% to 100% SPY (help from my brother who is a programmer)
best_score = -np.inf 
best_share = None

for share in np.linspace(0, 1, 101):
    w = np.array([share, 1 - share])
    score = phg.fitness(w, two_fund)          
    if score > best_score:                          
        best_score = score
        best_share = share

print('brute force best: SPY', round(best_share, 2), 'TLT', round(1 - best_share, 2), 'Sharpe', round(best_score, 4))

brute force best: SPY 0.43 TLT 0.57 Sharpe 1.0038


In [26]:
#compares the brute force method with the ga
ga_w, ga_hist = phg.run_ga(two_fund, pop_size=30, n_keep=6, generations=30, rate=0.1, seed=6449)

print('         GA: SPY', round(ga_w[0], 2), 'TLT', round(ga_w[1], 2), 'Sharpe', round(ga_hist[-1], 4))
print('brute force: SPY', round(best_share, 2), 'TLT', round(1 - best_share, 2), 'Sharpe', round(best_score, 4))
print('PASS' if abs(ga_w[0] - best_share) <= 0.02 else 'FAIL')

         GA: SPY 0.43 TLT 0.57 Sharpe 1.0039
brute force: SPY 0.43 TLT 0.57 Sharpe 1.0038
PASS
